# Tier 3 — 1/3: seleção dos Transformers (Optuna, 20 trials por modelo × base)

Seleção única por base: TPE (semente 0), hold-out 80/20 do treino da semente 0, `lr` na busca,
200 épocas / paciência 16. Retomável trial a trial.

**Como rodar:** Settings → Accelerator → **GPU T4 x2**, Internet **ligada** → **Save Version → Save & Run All**.
O runner para sozinho ao atingir 10 h (nenhum trial/semente novo começa depois disso) e tudo o que terminou
fica em `/kaggle/working/tier3_out/`, que vira a saída da versão salva.

**Se não terminar em uma sessão** (a última célula diz o que falta): crie uma nova versão com a saída da
anterior como entrada (**Add Input → Your Work → Notebook Output**) e Save & Run All de novo — a célula 4
retoma sozinha.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'

In [ ]:
# ── 2. Repositório ──
import os, subprocess, sys
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, '.')
assert 'TIER3_DEADLINE' in open('scripts/run_tier3.py').read(), 'clone sem prazo no runner — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (em cache no clone; nada é baixado) ──
subprocess.run('pip install -q einops optuna', shell=True)
from pathlib import Path
from src.data.loaders import DatasetLoader
for d in ['HIGGS500K', 'COVER', 'KDD99']:
    assert Path(f'data/raw/{d}.parquet').exists(), f'sem {d}.parquet no clone — refaça o pull'
    X, y, _ = DatasetLoader.load(d)
    print(f'{d:10s} {X.shape}  positivos {y.mean():.3f}')

In [ ]:
# ── 4. Configuração, retomada automática e execução nas duas placas ──
import json, shutil, time
from pathlib import Path

# Prazo: nenhum trial/semente novo começa depois de 10 h (margem para o trial em curso
# e para o Kaggle salvar a saída antes das 12 h). O que terminou fica em tier3_out/.
DEADLINE_H = 10.0
os.environ['TIER3_DEADLINE'] = str(time.time() + DEADLINE_H * 3600)

OUTDIR = Path('/kaggle/working/tier3_out'); OUTDIR.mkdir(exist_ok=True)
# Retomada: copia qualquer tier3_* das entradas (saída de uma sessão anterior
# adicionada via "Add Input" → Notebook Output). Nada a configurar.
for f in sorted(Path('/kaggle/input').rglob('tier3_*')):
    if f.is_file() and not (OUTDIR / f.name).exists():
        shutil.copy(f, OUTDIR / f.name); print('retomado:', f)

DATASETS = 'HIGGS500K COVER KDD99'
SEEDS    = list(range(10))
# SAINT (atenção B×B) é o mais caro: fica com o FT-CUR; os quatro FT dividem a outra placa.
GPU_MODELS = {0: ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax', 'FTTransformer_sparsemax'],
              1: ['SAINTColnorm', 'FTTransformerCURRandom']}

def files(gpu, tag=''):
    return (f'--selection-file {OUTDIR}/tier3_selection{tag}_gpu{gpu}.json '
            f'--final-file {OUTDIR}/tier3_final{tag}_gpu{gpu}.json '
            f'--storage sqlite:///{OUTDIR}/tier3_optuna{tag}_gpu{gpu}.db')

def run_2gpu(make_cmd, name):
    """Um processo por placa, log próprio em tier3_out/; espera os dois."""
    procs = []
    for gpu in (0, 1):
        log = open(OUTDIR / f'tier3_{name}_gpu{gpu}.log', 'a')
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} python -u ' + make_cmd(gpu)
        print(f'[gpu{gpu}]', cmd, flush=True)
        procs.append((gpu, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT), log))
    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(300)
        print(f'{(time.time()-t0)/3600:.2f} h —', ' | '.join(
            f'gpu{g}: ' + ((open(l.name).read().strip().splitlines() or [''])[-1][-110:]) for g, _, l in procs), flush=True)
    for g, p, l in procs:
        l.close(); print(f'gpu{g} terminou com código {p.returncode}')

In [ ]:
# ── 5. Seleção ──
run_2gpu(lambda g: 'scripts/run_tier3.py select --variants ' + ' '.join(GPU_MODELS[g]) +
         ' --datasets ' + DATASETS + ' ' + files(g), 'select')

In [ ]:
# ── 6. Situação ──
falta = []
for g in (0, 1):
    f = OUTDIR / f'tier3_selection_gpu{g}.json'
    sel = json.load(open(f)) if f.exists() else {}
    for v in GPU_MODELS[g]:
        for d in DATASETS.split():
            r = sel.get(v, {}).get(d)
            if r and r.get('complete'):
                print(f'OK    {v:26s} {d:10s} F1val {r["best_val_f1"]:.4f}  {r["params"]}')
            else:
                falta.append((v, d)); print(f'FALTA {v:26s} {d}')
print('\nSELEÇÃO COMPLETA — siga para tier3_2_final.' if not falta else
      f'\n{len(falta)} pendentes — rode esta sessão de novo com esta saída como entrada.')